# Text Retrieval and Summarizer ChatBot Framework 

### RAG (Retrieval-Augmented Generation) System

**Project Summary**:

This project is a text retrieval and summarization system that allows users to input a question and receive a concise summary based on relevant content.

It works by first converting the user’s input into numerical embeddings using a sentence transformer model. These embeddings are then compared against a pre-built vector index (FAISS) to identify the most relevant text chunks from your dataset. The retrieved content is combined and passed to a transformer-based summarization model BART, which generates a concise summary as the final output.

The entire pipeline is integrated into an interactive user interface built using Gradio, allowing users to easily input queries and view summarized results in real time.

Steps:
1. Retrieves relevant text from a User's Document (FAISS)
2. Converts Corpus to Sentences (Sentence Transformer)
3. Generates a Summarized output (HuggingFace Text Summarizer)

**Use of SBERT**:

Sentence Transformers(SBERT), uses pretrained "Embedding" models, all we do is provide them our chunks from previous step and it creates vectors. (huggingface)
Embeddings are dense, lower-dimensional, numerical vector representations of data such as text, images, or audio that capture semantic meaning and relationships.(soucre: google)

Steps:
1. Load an embedding model
2. Feed text chunks into the model
3. Convert each chunk into a vector of numbers

Transformer Model (all-MiniLM-L6-v2):
This is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.(huggingface)

**Use of FAISS**:

FAISS as a super-fast “vector search engine”, stands for Facebook AI Similarity Search. 
It is an open-source library developed by Meta's Fundamental AI Research group (formerly Facebook AI Research) designed for the efficient similarity search and clustering of dense vectors. (google)

Takes chunks of text from the document
As each chunk is previously converted to a 384-dimensional embedding by MiniLM
This store all embeddings in FAISS
so when a user asks a question, the question is converted to a vector and FAISS finds the nearest embeddings (most similar chunks of text from the document)
Then we pass those chunks to your LLM to generate the answer


**Final Pipeline**: 
Take PDF -> Get chunks -> Make embeddings -> Ask Question -> Retrieve Answer -> Summarize Result and Display Metrics

*--by Murk Asad*

In [ ]:
#%pip install langchain
#%pip install langchain_text_splitters
#%pip install sentence_transformers
#!pip install faiss-cpu
#%pip install transformers==4.41.2 sentencepiece
#%pip install tokenizers==0.20.3
#%pip install gradio
#%pip install torch
#%pip install matplotlib
#%pip install tempfile
#%pip install os
#%pip install re
#%pip install pdfplumber

In [3]:
import pdfplumber #intelligent that pypdf2 in handling complex pdfs like books with equations etc.
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
import gradio as gr
import matplotlib.pyplot as plt
import tempfile #using this library to save plots temporarily in local appdata
import os
import uuid  #this works along with tempfile to generate random but unique file names temporarily

corpus = ""

with pdfplumber.open("Deep+Learning+Ian+Goodfellow.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            corpus += text + " "

# Clean spacing
corpus = re.sub(r'\s+', ' ', corpus)

#Fix merged words
corpus = re.sub(r'([a-z])([A-Z])', r'\1 \2', corpus)

corpus = corpus.lower()

#Preprocessing Skipped(Reason):
#initially removed punctuations but it seems not very helpful for summarizer, so removed punctuations step forbetter answers
#Didnt remove stopwords because these stop words would help later while generating summarized text and context
#Lemmatization: tried and tested but doesnot do well, eats up main words and is not suitable for generating answer

# LANGCHAIN BASED CHUNKING 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50 
)
chunks = splitter.split_text(corpus)

# SENTENCE TRANSFORMER EMBEDDINGS 
model = SentenceTransformer("all-MiniLM-L6-v2")#free but slow
embeddings = model.encode(chunks)
dimension = embeddings.shape[1]

# FAISS 
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

# TEXT SUMMARIZER PIPELINE
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn"
)

# GRAPH PLOT FUNCTIONS - INLINE IN GRADIO
def create_graphs(chunk_lengths, retrieved_len, summary_len, stage_times):
    temp_dir = tempfile.gettempdir()
    uid = str(uuid.uuid4())

    # Graph 1: Stage times
    plt.figure(figsize=(8, 5))
    plt.bar(stage_times.keys(), stage_times.values())
    plt.xticks(rotation=30)
    plt.title("Pipeline Stage Execution Time", fontsize=14)
    plt.xlabel("Pipeline Stage")
    plt.ylabel("Time (seconds)")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()

    g1 = os.path.join(temp_dir, f"g1_{uid}.png")
    plt.savefig(g1)
    plt.close()

    # Graph 2: Chunk distribution
    plt.figure(figsize=(8, 5))
    plt.hist(chunk_lengths, bins=20)
    plt.title("Chunk Length Distribution", fontsize=14)
    plt.xlabel("Chunk Length (words)")
    plt.ylabel("Frequency")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()

    g2 = os.path.join(temp_dir, f"g2_{uid}.png")
    plt.savefig(g2)
    plt.close()

    # Graph 3: Text length comparison
    plt.figure(figsize=(6, 5))
    plt.bar(["Retrieved Text", "Summary"], [retrieved_len, summary_len])
    plt.title("Retrieved vs Summary Length", fontsize=14)
    plt.xlabel("Text Type")
    plt.ylabel("Word Count")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()

    g3 = os.path.join(temp_dir, f"g3_{uid}.png")
    plt.savefig(g3)
    plt.close()

    return g1, g2, g3

#  MAIN FUNCTION 
def chat_answer(message, history):
    import time
    t0 = time.time()
    #all t variables from this point are for graphs only

    try:
        # GET CONTEXT FROM HISTORY 
        context = " ".join(
            str(h["content"])
            for h in history[-3:]
            if h["role"] == "user"
        )
        # take last 3 messages from chat history
        # for h in history[-3:]  means looop through each message (h)
        # if h["role"] == "user" means keep only user messages (ignore assistant replies)
        # h["content"] will extract the actual text of the user question
        # then create a list of those questions and join them

        full_query = context + " " + message

        # QUERY EMBEDDING - EMBED THE QUESTION
        t1 = time.time()
        query_embeddings = model.encode([full_query])
        t2 = time.time()

        # ANSWER RETRIEVAL - GET BEST MATCH CLOSEST ANSWER FROM EMBEDDINGS
        distances, indices = index.search(np.array(query_embeddings).astype("float32"), k=2) #get 2 chunks only
        answer = " ".join([chunks[i] for i in indices[0]])
        t3 = time.time()

        # LIMIT TEXT - SAVE MEMORY
        answer = " ".join(answer.split()[:200])

        # SUMMARIZATION
        summary = summarizer(
            answer,
            repetition_penalty=5.0,
            length_penalty=0.3,
            min_length=20,
            max_length=50
        )
        t4 = time.time()

        summary_text = summary[0]["summary_text"] #pipeline returns alot of type of dictionaries, we only need the short summary from it so we use [0] and "summary_text"

        # GET METRICS
        retrieved_len = len(answer.split())
        summary_len = len(summary_text.split())

        #to fetch time it takes for evry step in the pipleine
        stage_times = {
            "Embed": t2 - t1,
            "Retrieve": t3 - t2,
            "Summarize": t4 - t3
        }

        chunk_lengths = [len(c.split()) for c in chunks]

        # GET GRAPHS FROM GRAPH FUNCTION - SENDING METRICS TO FUNCTION
        g1, g2, g3 = create_graphs(
            chunk_lengths,
            retrieved_len,
            summary_len,
            stage_times
        )

        metrics = f"""
                    Retrieved words: {retrieved_len}
                    Summary words: {summary_len}
                    Compression ratio: {round(summary_len / max(retrieved_len,1), 3)}

                    Embedding time: {round(stage_times['Embed'],3)}s
                    Retrieval time: {round(stage_times['Retrieve'],3)}s
                    Summarization time: {round(stage_times['Summarize'],3)}s
                    """

        return summary_text, metrics, g1, g2, g3

    except Exception as e:
        return f"Error: {str(e)}", "", None, None, None #get the 5 things from return statement or give None for properly handling error 

# GRADIO BASED UI
light_css = """
footer {visibility: hidden;}
.footer-note {
    text-align: center;
    color: #555;
    font-size: 14px;
    margin-top: 10px;
}
"""
with gr.Blocks() as demo:
    gr.Markdown("## Deep Learning Chat with Metrics & Graphs")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask any question about Machine Learning or Deep Learning")

    metrics_box = gr.Textbox(label="Metrics")

    g1 = gr.Image(label="Graph 1: Stage Times")
    g2 = gr.Image(label="Graph 2: Chunk Distribution")
    g3 = gr.Image(label="Graph 3: Length Comparison")

    def respond(message, history):
        answer, metrics, g1_img, g2_img, g3_img = chat_answer(message, history)
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": answer})
        return history, metrics, g1_img, g2_img, g3_img

    msg.submit(respond, [msg, chatbot], [chatbot, metrics_box, g1, g2, g3])
    gr.Markdown("<div class='footer-note'>RAG Project by Murk Asad</div>")

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://97d110cf7fddfb219c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


*The end*